In [32]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langchain_openrouter import ChatOpenRouter
from pydantic import BaseModel, Field
from dotenv import load_dotenv

In [33]:
load_dotenv()

llm_model = ChatOpenRouter(model="openai/gpt-oss-120b:free")
# llm_model = ChatOpenRouter(model="meta-llama/llama-3-8b-instruct")

In [34]:
class SentimentSchema(BaseModel):
    positive_sentiment: bool = Field(description="Sentiment of the given input. 'True' if positive. 'False' if negative.")

class ReviewAnalysisSchema(BaseModel):
    issue_type: str = Field(description="What exactly is the type of issue the user is addressing?")
    tone_of_user: str = Field(description="Tone used by user in the review.")
    urgency: str = Field(description="If user showing the urgency or not.")

In [35]:
st_llm1 = llm_model.with_structured_output(SentimentSchema)
st_llm2 = llm_model.with_structured_output(ReviewAnalysisSchema)

In [36]:
class ReviewState(TypedDict):
    review: str
    positive_sentiment: bool
    issue_type: str
    tone_of_user: str
    urgency: str
    response: str

In [37]:
def find_sentiment(state: ReviewState):
    review = state['review']

    prompt = f'''
Analyse the review given by user and return whether its sentiment is positive or not.
Review: {review}
'''
    
    positive_sentiment = st_llm1.invoke(prompt)

    state['positive_sentiment'] = positive_sentiment

    return state

In [38]:
def positive_response(state: ReviewState):
    review = state['review']

    prompt = f'''
Generate an appropriate response for the below review given by a user.
Review: {review}
'''

    response = llm_model.invoke(prompt).content

    return {'response': response}

In [39]:
def run_diagnosis(state: ReviewState):
    review = state['review']

    prompt = f'''
Analyze the review given by a user and tell about the issue type, tone of user and urgency.
Review: {review}
'''

    output = st_llm2.invoke(prompt)

    return {'issue_type': output.issue_type, 'tone_of_user': output.tone_of_user, 'urgency': output.urgency}

def negative_response(state: ReviewState):
    review = state['review']
    issue_type = state['issue_type']
    tone_of_user = state['tone_of_user']
    urgency = state['urgency']

    prompt = f'''You're provided with a review given by a user along with the issue type, tone of user and if there is any urgency or not. Analyze it and generate an appropriate response.
- Review: {review}
- Issue type: {issue_type}
- Tone of user: {tone_of_user}
- Urgency: {urgency}
'''
    
    response = llm_model.invoke(prompt).content

    return {'response': response}

In [40]:
def check_condition(state: ReviewState) -> Literal['positive_response', 'run_diagnosis']:
    positive_sentiment = state['positive_sentiment']

    if positive_sentiment:
        return 'positive_response'
    else:
        return 'run_diagnosis'

In [41]:
graph = StateGraph(ReviewState)

graph.add_node('find_sentiment', find_sentiment)
graph.add_node('run_diagnosis', run_diagnosis)
graph.add_node('positive_response', positive_response)
graph.add_node('negative_response', negative_response)

graph.add_edge(START, 'find_sentiment')
graph.add_conditional_edges('find_sentiment', check_condition)
graph.add_edge('positive_response', END)
graph.add_edge('run_diagnosis', 'negative_response')
graph.add_edge('negative_response', END)

workflow = graph.compile()

In [43]:
review = '''This is the worst pizza I've ever eaten. I was very hungry and expecting it to be delicious. But it was just a waste of money. Will never recommend to someone.'''

input_state = {'review': review}
output_state = workflow.invoke(input_state)

print(output_state)
print('final response -> ', output_state['response'])

{'review': "This is the worst pizza I've ever eaten. I was very hungry and expecting it to be delicious. But it was just a waste of money. Will never recommend to someone.", 'positive_sentiment': SentimentSchema(positive_sentiment=False), 'response': '**Response:**\n\nHi [Customer Name],\n\nWe’re truly sorry to hear that your recent pizza experience fell far short of expectations. We understand how disappointing it is to be hungry and receive a meal that doesn’t deliver on flavor or value.\n\nYour feedback is important to us, and we’d like the opportunity to make things right. Please reach out to us at **[phone number]** or **[email address]** with your order details, and we’ll gladly arrange a replacement pizza or a full refund—whichever you prefer.\n\nThank you for taking the time to let us know. We hope to earn back your trust and prove that we can deliver the delicious pizza you were expecting.\n\nSincerely,  \n[Your Name]  \nCustomer Experience Team  \n[Restaurant Name]'}
final re